In [9]:
import pandas as pd
import numpy as np
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")


In [10]:
data = pd.read_csv("data/Crop_recommendation.csv")

print("\nShape:", data.shape)
print("\nColumns:\n", data.columns.tolist())


Shape: (2200, 8)

Columns:
 ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']


Missing & Duplicate Analysis

In [11]:
print("\nMissing Values:\n", data.isnull().sum())
print("\nDuplicate Rows:", data.duplicated().sum())

data = data.drop_duplicates().reset_index(drop=True)


Missing Values:
 N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

Duplicate Rows: 0


STATISTICAL SUMMARY

In [12]:
print(data.describe())
print("\nUnique Values per Column:\n")
print(data.nunique())

                 N            P            K  temperature     humidity  \
count  2200.000000  2200.000000  2200.000000  2200.000000  2200.000000   
mean     50.551818    53.362727    48.149091    25.616244    71.481779   
std      36.917334    32.985883    50.647931     5.063749    22.263812   
min       0.000000     5.000000     5.000000     8.825675    14.258040   
25%      21.000000    28.000000    20.000000    22.769375    60.261953   
50%      37.000000    51.000000    32.000000    25.598693    80.473146   
75%      84.250000    68.000000    49.000000    28.561654    89.948771   
max     140.000000   145.000000   205.000000    43.675493    99.981876   

                ph     rainfall  
count  2200.000000  2200.000000  
mean      6.469480   103.463655  
std       0.773938    54.958389  
min       3.504752    20.211267  
25%       5.971693    64.551686  
50%       6.425045    94.867624  
75%       6.923643   124.267508  
max       9.935091   298.560117  

Unique Values per Column:


In [13]:
print("\n========== CROP DISTRIBUTION ==========")
print(data["label"].value_counts())


========== CROP DISTRIBUTION ==========
label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram      100
lentil         100
pomegranate    100
banana         100
mango          100
grapes         100
watermelon     100
muskmelon      100
apple          100
orange         100
papaya         100
coconut        100
cotton         100
jute           100
coffee         100
Name: count, dtype: int64


OUTLIER DETECTION AND REMOVAL(IQR)

In [14]:
numeric_cols = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

Q1 = data[numeric_cols].quantile(0.25)
Q3 = data[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

clean_data = data[
    ~((data[numeric_cols] < (Q1 - 1.5 * IQR)) |
      (data[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)
]
print("Before:", data.shape)
print("After :", clean_data.shape)

Before: (2200, 8)
After : (1768, 8)


Feature & Target Split

In [15]:
X = clean_data.drop(columns=["label"])
y = clean_data["label"]

Encode Target

In [16]:

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("\nEncoded Classes:\n", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


Encoded Classes:
 {'banana': np.int64(0), 'blackgram': np.int64(1), 'chickpea': np.int64(2), 'coconut': np.int64(3), 'coffee': np.int64(4), 'cotton': np.int64(5), 'jute': np.int64(6), 'kidneybeans': np.int64(7), 'lentil': np.int64(8), 'maize': np.int64(9), 'mango': np.int64(10), 'mothbeans': np.int64(11), 'mungbean': np.int64(12), 'muskmelon': np.int64(13), 'orange': np.int64(14), 'papaya': np.int64(15), 'pigeonpeas': np.int64(16), 'pomegranate': np.int64(17), 'rice': np.int64(18), 'watermelon': np.int64(19)}


Train-Test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

ML PIPELINE

In [18]:
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        ))
    ]
)

Hyperparameter Optimization

In [19]:
param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [None, 20, 40],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("\nBest Parameters:\n", grid.best_params_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best Parameters:
 {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}


MODEL EVALUATION

In [20]:
y_pred = best_model.predict(X_test)

print("\n========== MODEL PERFORMANCE ==========")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


========== MODEL PERFORMANCE ==========
Accuracy: 0.9915254237288136

Classification Report:

              precision    recall  f1-score   support

      banana       1.00      1.00      1.00        20
   blackgram       1.00      1.00      1.00        20
    chickpea       1.00      1.00      1.00        11
     coconut       1.00      1.00      1.00        17
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
        jute       0.91      1.00      0.95        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      0.95      0.97        20
       maize       1.00      1.00      1.00        20
       mango       1.00      1.00      1.00        20
   mothbeans       0.92      1.00      0.96        12
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        17
      papaya       1.00      1.00      1

FEATURE IMPORTANCE

In [21]:
importances = best_model.named_steps["model"].feature_importances_
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)
print(feature_importance)

       Feature  Importance
6     rainfall    0.221884
4     humidity    0.220707
2            K    0.172895
1            P    0.132766
0            N    0.114390
3  temperature    0.080434
5           ph    0.056924


In [22]:
joblib.dump(best_model, "crop_classifier.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

['label_encoder.pkl']

In [23]:
def predict_crop(N, P, K, temperature, humidity, ph, rainfall):
    model = joblib.load("crop_classifier.pkl")
    encoder = joblib.load("label_encoder.pkl")

    input_data = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    encoded_pred = model.predict(input_data)[0]
    return encoder.inverse_transform([encoded_pred])[0]

SAMPLE PREDICTION

In [24]:
if __name__ == "__main__":
    result = predict_crop(
        N=90,
        P=42,
        K=43,
        temperature=20.87,
        humidity=82.00,
        ph=6.50,
        rainfall=202.93
    )
    print("\nPredicted Crop:", result)



Predicted Crop: rice
